# 07 · Model Registry and deployment (batch + real-time)
**AAI-540 · Group 4 · Criteo CTR**

Takes the tuned artefact from notebook 06 and puts it into service:

1. **Register** it as a versioned model package with its evaluation metrics and
   a model card attached, in `PendingApproval`.
2. **Approve** it explicitly — a separate, recorded action, not a flag on the
   registration call.
3. **Batch Transform** over the held-back production split. This is the
   **primary** serving mode for the project: scoring a large held-out set is
   throughput-bound and embarrassingly parallel, and every number the design
   document reports comes from here.
4. **Real-time endpoint** with data capture enabled. This is the mode the
   business case actually needs (a bid decision cannot wait for a batch job)
   and the mode the monitors in notebook 08 watch.
5. **Parity check** between the two paths.

**Why the approval gate is manual.** This model sets bid prices. An automatic
promotion path trades a human review of the metrics, the bias caveats and the
model card for a convenience nobody asked for. The registration therefore
lands in `PendingApproval` and a person moves it.

**Why parity is checked rather than assumed.** Both paths load the same
artefact, but they encode requests differently. If they disagree, every
monitoring number computed from the endpoint is wrong for the batch job and
vice versa — and the disagreement is invisible until someone compares them.
`deployment.parity_report` is that comparison.

In [ ]:
import os, sys, json, time
sys.path.insert(0, os.path.abspath("../src"))

import boto3
import numpy as np
import pandas as pd
import sagemaker
from sagemaker.model import Model

from criteo_ctr import config as C
from criteo_ctr import deployment as Dep
from criteo_ctr import evaluation as E
from criteo_ctr import features as F
from criteo_ctr import monitoring as M
from criteo_ctr.io_utils import Store, s3_uri, canonical_columns

sess = sagemaker.Session()
boto_sess = sess.boto_session
sm = boto_sess.client("sagemaker")
runtime = boto_sess.client("sagemaker-runtime")
region = sess.boto_region_name
role = sagemaker.get_execution_role()
bucket = C.BUCKET or sess.default_bucket()
store = Store(bucket, boto_sess)

MODEL_PACKAGE_GROUP = "criteo-ctr-models"
ENDPOINT_NAME = "criteo-ctr-endpoint"
CAPTURE_PREFIX = f"{C.PREFIX}/monitoring/data-capture"
PREDICTIONS_PREFIX = f"{C.PREFIX}/predictions"

record = json.loads(boto_sess.client("s3").get_object(
    Bucket=bucket, Key=f"{C.ARTIFACTS_PREFIX}/model_record.json")["Body"].read())
print(f"artifact : {record['model_artifact']}")
print(f"test log loss {record['evaluation']['binary_classification_metrics']['log_loss']['value']:.6f}")

## 1 · Model package group and registration

In [ ]:
try:
    sm.create_model_package_group(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelPackageGroupDescription=(
            "Criteo display-ad CTR models. Versions carry test-split metrics and a "
            "model card; only Approved versions are deployed."))
    print(f"created group {MODEL_PACKAGE_GROUP}")
except sm.exceptions.ResourceInUse:
    print(f"group {MODEL_PACKAGE_GROUP} already exists")

model = Model(image_uri=record["image_uri"], model_data=record["model_artifact"],
              role=role, sagemaker_session=sess)

from sagemaker.model_metrics import MetricsSource, ModelMetrics
model_metrics = ModelMetrics(model_statistics=MetricsSource(
    s3_uri=s3_uri(bucket, f"{C.PREFIX}/models/evaluation/evaluation.json"),
    content_type="application/json"))

package = model.register(
    content_types=["text/csv"], response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.large"],
    transform_instances=["ml.m5.large", "ml.m5.xlarge"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    approval_status="PendingApproval",          # a human promotes it, see below
    model_metrics=model_metrics,
    description=f"XGBoost CTR model from tuning job {record['tuning_job']}",
)
package_arn = package.model_package_arn
print(f"registered: {package_arn}")
print(f"status    : {sm.describe_model_package(ModelPackageName=package_arn)['ModelApprovalStatus']}")

## 2 · Attach the model card

The card travels with the model package, which is the point: the fairness
limitation is a property of the artefact, not of a document somebody may or may
not read. `deployment.model_card_content` states plainly that a protected-class
audit is **not possible** on hashed features and is therefore out of scope —
rather than reporting a fairness check that was never run.

In [ ]:
card = Dep.model_card_content(
    metrics=record["evaluation"]["binary_classification_metrics"],
    model_package_arn=package_arn,
    dataset_uri=s3_uri(bucket, f"{C.SPLITS_PREFIX}/xgboost/train/"))

card_key = f"{C.ARTIFACTS_PREFIX}/model_card.json"
store.put_json(card, card_key)

try:
    sm.create_model_card(
        ModelCardName=f"criteo-ctr-{int(time.time())}",
        Content=json.dumps(card), ModelCardStatus="Review")
    print("model card created in SageMaker Model Cards")
except Exception as exc:
    # Model Cards is not available in every account/region; the JSON artefact
    # is the source of truth either way.
    print(f"Model Cards API unavailable ({type(exc).__name__}); card saved to S3")

print(f"card: s3://{bucket}/{card_key}")
print("\nethical considerations recorded with the model:")
print(card["additional_information"]["ethical_considerations"][:400], "...")

## 3 · Approve the version

Deliberately its own cell, so the approval is a visible, timestamped action in
the notebook and in the registry's history — and so the demonstration in the
video can show the version moving from `PendingApproval` to `Approved`.

In [ ]:
sm.update_model_package(ModelPackageArn=package_arn, ModelApprovalStatus="Approved",
                        ApprovalDescription=(
                            "Beats the global-CTR benchmark on held-out test log loss; "
                            "calibration and bias caveats reviewed in the model card."))
approved = Dep.latest_approved_package(sm, MODEL_PACKAGE_GROUP)
print(f"approved version {approved['ModelPackageVersion']} "
      f"({approved['ModelPackageArn'].split('/')[-1]})")

versions = sm.list_model_packages(ModelPackageGroupName=MODEL_PACKAGE_GROUP,
                                  SortBy="CreationTime", SortOrder="Descending")
display(pd.DataFrame([{ "version": v["ModelPackageVersion"],
                        "status": v["ModelApprovalStatus"],
                        "created": v["CreationTime"]}
                      for v in versions["ModelPackageSummaryList"]]))

## 4 · Batch Transform over the production split — the primary serving path

The input is `production_features.csv`: the most recent 40% of the window,
**without labels**. The labels for these rows stay in
`splits/ground_truth/production_labels.parquet` and are only used by the
monitors in notebook 08, which is what makes the production split a credible
stand-in for live traffic.

In [ ]:
batch_input = s3_uri(bucket, f"{C.SPLITS_PREFIX}/xgboost/production/production_features.csv")
batch_output = s3_uri(bucket, f"{PREDICTIONS_PREFIX}/batch/")

deployable = Model(image_uri=record["image_uri"], model_data=record["model_artifact"],
                   role=role, sagemaker_session=sess, name=f"criteo-ctr-{int(time.time())}")

t0 = time.time()
transformer = Dep.run_batch_transform(deployable, batch_input, batch_output,
                                      instance_type="ml.m5.large", instance_count=1)
print(f"batch transform finished in {time.time() - t0:.0f}s")
print(f"job    : {transformer.latest_transform_job.name}")
print(f"output : {batch_output}")

In [ ]:
# Align the predictions to record ids. The join is positional and only valid
# because the input CSV was written from the ordered production frame -
# batch_output_to_frame asserts the lengths match rather than trusting that.
production = canonical_columns(
    store.read_parquet_prefix(f"{C.SPLITS_PREFIX}/parquet/split=production/"))
production = production.sort_values(C.RECORD_ID).reset_index(drop=True)

out_keys = [k for k in store.list_keys(f"{PREDICTIONS_PREFIX}/batch/") if k.endswith(".out")]
body = "".join(boto_sess.client("s3").get_object(Bucket=bucket, Key=k)["Body"]
               .read().decode() for k in sorted(out_keys))
batch_predictions = Dep.batch_output_to_frame(body, production[C.RECORD_ID].tolist())

print(f"{len(batch_predictions):,} predictions aligned to record ids")
print(batch_predictions.head().to_string(index=False))
print(f"\npredicted CTR {batch_predictions['prediction'].mean():.5f}")
store.put_parquet(batch_predictions, f"{PREDICTIONS_PREFIX}/aligned/production_predictions.parquet")

### Scored against the held-back labels

This is the project's headline measurement: the model scoring traffic it has
never seen, from a period after everything it was trained on.

In [ ]:
labels = store.read_parquet_prefix(f"{C.SPLITS_PREFIX}/ground_truth/")
labels = canonical_columns(labels)[[C.RECORD_ID, C.LABEL]]
scored = M.join_predictions_to_labels(batch_predictions, labels)

production_report = E.evaluate(scored[C.LABEL], scored["prediction"], name="production_batch")
test_metrics = record["evaluation"]["binary_classification_metrics"]

summary = pd.DataFrame([
    {"split": "test (notebook 06)", "log_loss": test_metrics["log_loss"]["value"],
     "roc_auc": test_metrics["auc"]["value"],
     "calibration_error": test_metrics["expected_calibration_error"]["value"]},
    {"split": "production (batch)", "log_loss": production_report["log_loss"],
     "roc_auc": production_report["roc_auc"],
     "calibration_error": production_report["expected_calibration_error"]},
]).set_index("split")
display(summary.round(5))

print(f"normalised entropy on production: {production_report['normalised_entropy']:.4f}")
print("A production score materially worse than test is drift, not a bug - it is\n"
      "exactly what the monitors in notebook 08 are built to detect and report.")
store.put_json(production_report, f"{C.ARTIFACTS_PREFIX}/production_batch_report.json")

## 5 · Real-time endpoint with data capture

Capture is set to 100% of requests. That is right here and would be wrong at
bidding volume: this endpoint serves validation traffic in short bursts, and a
sampled capture would leave the monitoring windows too thin to compute a stable
log loss.

In [ ]:
predictor = deployable.deploy(
    initial_instance_count=1, instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=Dep.data_capture_config(
        destination_s3_uri=s3_uri(bucket, CAPTURE_PREFIX), sampling_percentage=100),
    wait=True)
print(f"endpoint {ENDPOINT_NAME} in service")
print(f"capture  -> s3://{bucket}/{CAPTURE_PREFIX}")

described = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print(f"status {described['EndpointStatus']}, "
      f"variant {described['ProductionVariants'][0]['VariantName']}, "
      f"instance {described['ProductionVariants'][0]['CurrentInstanceCount']}x"
      f"{described['ProductionVariants'][0].get('CurrentServingInstanceType', 'ml.m5.large')}")

In [ ]:
# Invoke with the first slice of the production split. InferenceId is set to the
# first record_id of each call, which is what lets the model-quality monitor
# join these predictions back to the held-back labels later.
sample = production.head(2_000)
t0 = time.time()
realtime = Dep.invoke_rows(runtime, ENDPOINT_NAME, sample, batch_size=500)
elapsed = time.time() - t0
print(f"{len(realtime):,} rows scored in {elapsed:.1f}s "
      f"({len(realtime) / elapsed:,.0f} rows/s across {int(np.ceil(len(sample)/500))} calls)")
display(realtime.head())

## 6 · Parity: do the two serving paths agree?

Both paths loaded the same artefact and were sent the same rows. The tolerance
sits below the precision of the CSV wire format rather than at it — comparing
to `1e-6` when the format carries six decimals would fail on rounding alone.

In [ ]:
merged = realtime.merge(batch_predictions, on=C.RECORD_ID,
                        suffixes=("_realtime", "_batch"))
parity = Dep.parity_report(merged["prediction_batch"], merged["prediction_realtime"],
                           tolerance=1e-5)
print(json.dumps(parity, indent=2))

if parity["passed"]:
    print("\nPASS - batch and real-time return the same scores, so every monitoring\n"
          "number computed from the endpoint also describes the batch path.")
else:
    print("\nFAIL - the paths disagree. Do not trust monitoring until the request\n"
          "encoding is reconciled; check column order and CSV formatting first.")
store.put_json(parity, f"{C.ARTIFACTS_PREFIX}/inference_parity.json")

In [ ]:
# Confirm capture is landing in S3 before notebook 08 depends on it.
time.sleep(60)
capture_keys = store.list_keys(f"{CAPTURE_PREFIX}/{ENDPOINT_NAME}/")
print(f"{len(capture_keys)} capture object(s)")
for k in capture_keys[:3]:
    print("  ", k)

if capture_keys:
    first = boto_sess.client("s3").get_object(
        Bucket=bucket, Key=capture_keys[0])["Body"].read().decode()
    parsed = M.flatten_capture_records(first.strip().split("\n"))
    print(f"\nparsed {len(parsed)} predictions out of "
          f"{len(first.strip().splitlines())} captured invocation(s)")
    display(parsed.head())
else:
    print("capture can lag a few minutes; re-run this cell before notebook 08")

## 7 · Deployment record

The endpoint is left running for notebook 08 (the monitors need it) and
notebook 09 (the CloudWatch dashboard needs metrics). **Delete it when the
demonstration is recorded** — the last cell is the teardown.

In [ ]:
deployment_record = {
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "model_package_arn": package_arn,
    "model_package_group": MODEL_PACKAGE_GROUP,
    "approval_status": "Approved",
    "endpoint_name": ENDPOINT_NAME,
    "endpoint_instance": "ml.m5.large",
    "data_capture_s3": s3_uri(bucket, CAPTURE_PREFIX),
    "batch_transform_job": transformer.latest_transform_job.name,
    "batch_output": batch_output,
    "batch_rows": int(len(batch_predictions)),
    "production_report": {k: production_report[k] for k in
                          ("log_loss", "normalised_entropy", "roc_auc", "pr_auc",
                           "expected_calibration_error", "observed_ctr", "mean_predicted")},
    "inference_parity": parity,
    "model_card_s3": s3_uri(bucket, card_key),
}
store.put_json(deployment_record, f"{C.ARTIFACTS_PREFIX}/deployment_record.json")
with open(os.path.abspath("../artifacts/deployment_record.json"), "w") as fh:
    json.dump(deployment_record, fh, indent=2, default=str)
print(json.dumps(deployment_record, indent=2, default=str)[:900])

In [ ]:
# TEARDOWN - run after the monitoring notebooks and the video recording.
# An idle endpoint bills by the hour; leaving it up is the single easiest way
# to exhaust a Learner Lab budget.
TEARDOWN = False
if TEARDOWN:
    sagemaker.Session().delete_endpoint(ENDPOINT_NAME)
    sagemaker.Session().delete_endpoint_config(ENDPOINT_NAME)
    print(f"deleted endpoint {ENDPOINT_NAME}")
else:
    print(f"{ENDPOINT_NAME} left running for notebooks 08 and 09.\n"
          "Set TEARDOWN = True and re-run this cell when finished.")